# MScFE 620 Group Work Project 1

## Step 1: Put-Call Parity & Binomial Trees

**Q1: Does put-call parity apply to European options?**
Yes, it does. Put-call parity is the core relationship for European options. It shows that buying a call and shorting a put gives you the same payoff as a forward contract. Since European options can only be exercised at expiration, we can lock this in without worrying about early exercise breaking the math.

**Q2: Solve for Call**
Rearranging the standard put-call parity formula ($C - P = S_0 - K e^{-rT}$) to solve for the Call price ($C$), we get:
$C = P + S_0 - K e^{-rT}$

**Q3: Solve for Put**
Similarly, isolating the Put price ($P$) gives us:
$P = C - S_0 + K e^{-rT}$

**Q4: Does it apply to American options?**
Not exactly. Because American options can be exercised early, the strict equality breaks. Instead, American options follow this inequality:
$S_0 - K \leq C - P \leq S_0 - K e^{-rT}$
This means the price difference between an American call and put has to stay within these bounds to prevent arbitrage.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global Parameters
S0 = 100.0
K = 100.0
r = 0.05
sigma = 0.20
T = 0.25  # 3 months
N_steps = 200

def binomial_pricer(S0, K, r, sigma, T, N, is_call=True, is_american=False):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    # Terminal prices
    j = np.arange(N + 1)
    ST = S0 * (u ** (N - j)) * (d ** j)

    # Payoffs
    if is_call:
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)

    V_up = V_down = 0.0

    # Step backward
    for step in range(N - 1, -1, -1):
        V = disc * (p * V[:-1] + (1.0 - p) * V[1:])

        if is_american:
            j_step = np.arange(step + 1)
            S_step = S0 * (u ** (step - j_step)) * (d ** j_step)

            if is_call:
                intrinsic = np.maximum(S_step - K, 0.0)
            else:
                intrinsic = np.maximum(K - S_step, 0.0)

            V = np.maximum(V, intrinsic)

        # Grab the step 1 nodes for Delta
        if len(V) == 2:
            V_up = V[0]
            V_down = V[1]

    # Delta
    S_up = S0 * u
    S_down = S0 * d
    delta = (V_up - V_down) / (S_up - S_down)

    return float(V[0]), float(delta)

# Q5, Q6: European Prices and Deltas
eu_call_price, eu_call_delta = binomial_pricer(S0, K, r, sigma, T, N_steps, is_call=True, is_american=False)
eu_put_price, eu_put_delta = binomial_pricer(S0, K, r, sigma, T, N_steps, is_call=False, is_american=False)

# Q7: Vega for +5% Volatility
eu_call_bump, _ = binomial_pricer(S0, K, r, sigma + 0.05, T, N_steps, is_call=True, is_american=False)
eu_put_bump, _ = binomial_pricer(S0, K, r, sigma + 0.05, T, N_steps, is_call=False, is_american=False)

eu_call_vega = eu_call_bump - eu_call_price
eu_put_vega = eu_put_bump - eu_put_price

eu_data = {
    'Metric': ['Price', 'Delta', 'Vega (+5% Vol)'],
    'European Call': [f"${eu_call_price:.2f}", f"{eu_call_delta:.2f}", f"${eu_call_vega:.2f}"],
    'European Put': [f"${eu_put_price:.2f}", f"{eu_put_delta:.2f}", f"${eu_put_vega:.2f}"]
}

display(pd.DataFrame(eu_data))



**Q5-Q7 Analysis**

Delta tells us how much the option price moves if the stock moves by $1. It is a good proxy for the probability of the option finishing in the money. Calls have a positive delta because their value goes up with the stock price. Puts have a negative delta because we make money when the stock goes down. Vega is positive for both because higher volatility just means a higher chance for big price swings, making both options more valuable.



In [ ]:
# Q8, Q9, Q10: American Options
am_call_price, am_call_delta = binomial_pricer(S0, K, r, sigma, T, N_steps, is_call=True, is_american=True)
am_put_price, am_put_delta = binomial_pricer(S0, K, r, sigma, T, N_steps, is_call=False, is_american=True)

# Vega for +5% Volatility
am_call_bump, _ = binomial_pricer(S0, K, r, sigma + 0.05, T, N_steps, is_call=True, is_american=True)
am_put_bump, _ = binomial_pricer(S0, K, r, sigma + 0.05, T, N_steps, is_call=False, is_american=True)

am_call_vega = am_call_bump - am_call_price
am_put_vega = am_put_bump - am_put_price

am_data = {
    'Metric': ['Price', 'Delta', 'Vega (+5% Vol)'],
    'American Call': [f"${am_call_price:.2f}", f"{am_call_delta:.2f}", f"${am_call_vega:.2f}"],
    'American Put': [f"${am_put_price:.2f}", f"{am_put_delta:.2f}", f"${am_put_vega:.2f}"]
}

display(pd.DataFrame(am_data))



**Q11-Q14: Comparing European vs. American**

For our European options, $C - P$ is $\$4.61 - \$3.37 = \$1.24$. $S_0 - K e^{-rT}$ is $\$100 - \$98.76 = \$1.24$. So, put-call parity holds perfectly.

For the American options, $C - P$ is $\$4.61 - \$3.48 = \$1.13$. This fits right inside the required bounds of $0 \leq \$1.13 \leq \$1.24$.

As we can see, the European and American calls are priced exactly the same. Since this stock pays no dividends, there is no reason to ever exercise a call early. The American put, however, is more expensive than the European put ($\$3.48$ vs $\$3.37$). If the stock drops hard, we can exercise the American put early, cash out the strike price, and earn interest on that cash. That early exercise right has value.

## Step 2: Trinomial Trees & Sensitivities



In [ ]:
def trinomial_pricer(S0, K, r, sigma, T, N, is_call=True, is_american=False):
    dt = T / N
    dx = sigma * np.sqrt(3.0 * dt)
    nu = r - 0.5 * sigma**2

    pu = 0.5 * ((sigma**2 * dt + nu**2 * dt**2) / dx**2 + (nu * dt) / dx)
    pd = 0.5 * ((sigma**2 * dt + nu**2 * dt**2) / dx**2 - (nu * dt) / dx)
    pm = 1.0 - pu - pd

    disc = np.exp(-r * dt)

    j_vals = np.arange(-N, N + 1)
    ST = S0 * np.exp(j_vals * dx)

    if is_call:
        V = np.maximum(ST - K, 0.0)
    else:
        V = np.maximum(K - ST, 0.0)

    for step in range(N - 1, -1, -1):
        n_nodes = 2 * step + 1
        V_new = np.zeros(n_nodes)

        for i in range(n_nodes):
            V_new[i] = disc * (pu * V[i + 2] + pm * V[i + 1] + pd * V[i])

        if is_american:
            j_step = np.arange(-step, step + 1)
            S_step = S0 * np.exp(j_step * dx)

            if is_call:
                intrinsic = np.maximum(S_step - K, 0.0)
            else:
                intrinsic = np.maximum(K - S_step, 0.0)

            V_new = np.maximum(V_new, intrinsic)

        V = V_new

    return float(V[0])

# Q15-Q18: 5 strikes
strikes = [90.0, 95.0, 100.0, 105.0, 110.0]
results = []

for k in strikes:
    eu_c = trinomial_pricer(S0, k, r, sigma, T, N_steps, is_call=True, is_american=False)
    eu_p = trinomial_pricer(S0, k, r, sigma, T, N_steps, is_call=False, is_american=False)
    am_c = trinomial_pricer(S0, k, r, sigma, T, N_steps, is_call=True, is_american=True)
    am_p = trinomial_pricer(S0, k, r, sigma, T, N_steps, is_call=False, is_american=True)

    results.append({
        'Strike': f"${int(k)}",
        'European Call': f"${eu_c:.2f}",
        'American Call': f"${am_c:.2f}",
        'European Put': f"${eu_p:.2f}",
        'American Put': f"${am_p:.2f}"
    })

display(pd.DataFrame(results))



**Pricing Trends**

As we move to higher strike prices, the calls get cheaper because they are moving out of the money, while the puts get more expensive since they move deeper in the money. We can also see that the American early-exercise premium for puts gets much larger for the deeper ITM strikes.



In [ ]:
# Q19-Q22: Generating Plots
S_range = np.linspace(70, 130, 61)
K_range = np.linspace(70, 130, 61)
K_fixed = 100.0
S_fixed = 100.0

eu_c_S = [trinomial_pricer(s, K_fixed, r, sigma, T, N_steps, True, False) for s in S_range]
eu_p_S = [trinomial_pricer(s, K_fixed, r, sigma, T, N_steps, False, False) for s in S_range]
am_c_S = [trinomial_pricer(s, K_fixed, r, sigma, T, N_steps, True, True) for s in S_range]
am_p_S = [trinomial_pricer(s, K_fixed, r, sigma, T, N_steps, False, True) for s in S_range]

eu_c_K = [trinomial_pricer(S_fixed, k, r, sigma, T, N_steps, True, False) for k in K_range]
am_c_K = [trinomial_pricer(S_fixed, k, r, sigma, T, N_steps, True, True) for k in K_range]
eu_p_K = [trinomial_pricer(S_fixed, k, r, sigma, T, N_steps, False, False) for k in K_range]
am_p_K = [trinomial_pricer(S_fixed, k, r, sigma, T, N_steps, False, True) for k in K_range]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plt.style.use('seaborn-v0_8-whitegrid')

# Q19
axes[0, 0].plot(S_range, eu_c_S, label='Call', color='blue')
axes[0, 0].plot(S_range, eu_p_S, label='Put', color='red')
axes[0, 0].set_title('European Options vs Stock Price')
axes[0, 0].set_xlabel('Stock Price ($)')
axes[0, 0].set_ylabel('Option Price ($)')
axes[0, 0].legend()

# Q20
axes[0, 1].plot(S_range, am_c_S, label='Call', color='blue', linestyle='--')
axes[0, 1].plot(S_range, am_p_S, label='Put', color='red', linestyle='--')
axes[0, 1].set_title('American Options vs Stock Price')
axes[0, 1].set_xlabel('Stock Price ($)')
axes[0, 1].set_ylabel('Option Price ($)')
axes[0, 1].legend()

# Q21
axes[1, 0].plot(K_range, eu_c_K, label='European Call', color='blue')
axes[1, 0].plot(K_range, am_c_K, label='American Call', color='lightblue', linestyle='--')
axes[1, 0].set_title('Call Options vs Strike Price')
axes[1, 0].set_xlabel('Strike Price ($)')
axes[1, 0].set_ylabel('Option Price ($)')
axes[1, 0].legend()

# Q22
axes[1, 1].plot(K_range, eu_p_K, label='European Put', color='red')
axes[1, 1].plot(K_range, am_p_K, label='American Put', color='salmon', linestyle='--')
axes[1, 1].set_title('Put Options vs Strike Price')
axes[1, 1].set_xlabel('Strike Price ($)')
axes[1, 1].set_ylabel('Option Price ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()



**Q23 & Q24: Put-Call Parity**

Checking the data from our tree, put-call parity holds perfectly as an exact equation for all 5 strikes on the European side. For the American options, the exact math breaks down but the prices sit cleanly inside the required inequality bounds we defined earlier.

## Step 3: Dynamic Delta Hedging



In [ ]:
# Hedging Parameters
S0_h = 180.0
K_h = 182.0
r_h = 0.02
sigma_h = 0.25
T_h = 0.5

def hedge_tree(S0, K, r, sigma, T, N, is_american=False):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    S = np.zeros((N + 1, N + 1))
    V = np.zeros((N + 1, N + 1))
    Delta = np.zeros((N, N + 1))

    for i in range(N + 1):
        for j in range(i + 1):
            S[i, j] = S0 * (u ** j) * (d ** (i - j))

    for j in range(N + 1):
        V[N, j] = max(K - S[N, j], 0)

    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            exp_val = disc * (p * V[i + 1, j + 1] + (1 - p) * V[i + 1, j])
            intr_val = max(K - S[i, j], 0) if is_american else 0

            V[i, j] = max(exp_val, intr_val)
            Delta[i, j] = (V[i + 1, j + 1] - V[i + 1, j]) / (S[i + 1, j + 1] - S[i + 1, j])

    return S, V, Delta, dt

def run_hedge(S, V, Delta, dt, N, K, r, path):
    records = []
    shares = 0.0
    cash = 0.0

    for step in range(N):
        node = path[step]
        spot = S[step, node]
        tgt_delta = Delta[step, node]

        trade = tgt_delta - shares
        cost = trade * spot

        if step == 0:
            cash = V[0, 0] - cost
        else:
            cash = cash * np.exp(r * dt) - cost

        shares = tgt_delta

        records.append({
            'Step': step,
            'Stock Price': f"${spot:.2f}",
            'Delta': f"{tgt_delta:.4f}",
            'Shares Traded': f"{trade:.4f}",
            'Cash Account': f"${cash:.2f}"
        })

    final_spot = S[N, path[N]]
    payoff = max(K - final_spot, 0)
    cash = cash * np.exp(r * dt) + shares * final_spot - payoff

    records.append({
        'Step': N,
        'Stock Price': f"${final_spot:.2f}",
        'Delta': "0.0000",
        'Shares Traded': f"{-shares:.4f}",
        'Cash Account': f"${cash:.2f}"
    })

    return pd.DataFrame(records)

# Q25: 3-step European Put
S_eu, V_eu, D_eu, dt_eu = hedge_tree(S0_h, K_h, r_h, sigma_h, T_h, 3, False)
eu_path = [0, 0, 0, 1]  # down, down, up

print("3-Step European Put Hedging")
display(run_hedge(S_eu, V_eu, D_eu, dt_eu, 3, K_h, r_h, eu_path))



In [ ]:
# Q26: 25-step American Put
S_am, V_am, D_am, dt_am = hedge_tree(S0_h, K_h, r_h, sigma_h, T_h, 25, True)

# Generate random path
np.random.seed(42)
p_up = (np.exp(r_h * dt_am) - (1 / np.exp(sigma_h * np.sqrt(dt_am)))) / (np.exp(sigma_h * np.sqrt(dt_am)) - (1 / np.exp(sigma_h * np.sqrt(dt_am))))
moves = np.random.rand(25) < p_up

am_path = [0]
for m in moves:
    am_path.append(am_path[-1] + int(m))

df_am = run_hedge(S_am, V_am, D_am, dt_am, 25, K_h, r_h, am_path)
print("25-Step American Put Hedging (First and Last 5 Steps)")
display(pd.concat([df_am.head(5), df_am.tail(5)]))



In [ ]:
# Q27: Asian ATM Put
N_a = 25
dt_a = T_h / N_a
u_a = np.exp(sigma_h * np.sqrt(dt_a))
d_a = 1.0 / u_a

# Grab the same asset prices we used for the American path
spot_path = [S0_h]
for i in range(N_a):
    spot_path.append(spot_path[-1] * (u_a if am_path[i+1] > am_path[i] else d_a))

def calc_asian(spot, history, K, r, sigma, dt, rem_steps, sims=50000):
    tot_steps = len(history) + rem_steps
    hist_sum = sum(history)

    if rem_steps == 0:
        avg = hist_sum / tot_steps
        return max(K - avg, 0), (-1.0 / tot_steps if avg < K else 0.0)

    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp(r * dt) - d) / (u - d)

    moves = np.random.rand(sims, rem_steps) < p
    paths = np.zeros((sims, rem_steps))
    paths[:, 0] = spot * np.where(moves[:, 0], u, d)

    for i in range(1, rem_steps):
        paths[:, i] = paths[:, i-1] * np.where(moves[:, i], u, d)

    fut_sums = paths.sum(axis=1)
    avgs = (hist_sum + fut_sums) / tot_steps
    payoffs = np.exp(-r * rem_steps * dt) * np.maximum(K - avgs, 0)
    val = payoffs.mean()

    # Delta via bump
    eps = spot * 0.001
    shift_sums = fut_sums * ((spot + eps) / spot)
    shift_avgs = (hist_sum + eps + shift_sums) / tot_steps
    shift_payoffs = np.exp(-r * rem_steps * dt) * np.maximum(K - shift_avgs, 0)
    val_shift = shift_payoffs.mean()

    return val, (val_shift - val) / eps

np.random.seed(42)
init_price, _ = calc_asian(S0_h, [S0_h], K_h, r_h, sigma_h, dt_a, N_a, sims=100000)

a_recs = []
shares_a = 0.0
cash_a = 0.0

for s in range(N_a):
    spot = spot_path[s]
    hist = spot_path[:s+1]

    _, delta = calc_asian(spot, hist, K_h, r_h, sigma_h, dt_a, N_a - s)
    trade = delta - shares_a
    cost = trade * spot

    if s == 0:
        cash_a = init_price - cost
    else:
        cash_a = cash_a * np.exp(r_h * dt_a) - cost

    shares_a = delta

    a_recs.append({
        'Step': s,
        'Stock Price': f"${spot:.2f}",
        'Delta': f"{delta:.4f}",
        'Shares Traded': f"{trade:.4f}",
        'Cash Account': f"${cash_a:.2f}"
    })

final_spot = spot_path[-1]
final_avg = sum(spot_path) / len(spot_path)
payoff = max(K_h - final_avg, 0)

cash_a = cash_a * np.exp(r_h * dt_a) + shares_a * final_spot - payoff
a_recs.append({
    'Step': N_a,
    'Stock Price': f"${final_spot:.2f}",
    'Delta': "0.0000",
    'Shares Traded': f"{-shares_a:.4f}",
    'Cash Account': f"${cash_a:.2f}"
})

df_asian = pd.DataFrame(a_recs)
print("25-Step Asian ATM Put Hedging (First and Last 5 Steps)")
display(pd.concat([df_asian.head(5), df_asian.tail(5)]))



**Dynamic Hedging Comparison**

When we look at hedging these three options, the capital and risk requirements are completely different.

The European put is standard. We just adjust our hedge as the spot price moves, and we know exactly when the liability hits—at expiration.

The American put is much harder to manage. Because we can get assigned early if the stock tanks, we have to keep way more cash on hand. The delta moves aggressively when it drops deep in the money, forcing us to buy a lot of shares fast to stay neutral, which eats up cash and drives up transaction costs.

The Asian put is by far the easiest and cheapest to hedge. Since it settles on the average price, single-day price spikes near expiration don't really move the needle. We don't have to trade as many shares to stay hedged, our cash flow stays pretty stable, and the overall premium is cheaper because the risk is smoothed out.

